# 00 - Setup: Carga de Dados no SQL Server

Este notebook cria o banco **LojaDB** e carrega os arquivos CSV da pasta `data/` para quatro tabelas relacionais: `clientes`, `produtos`, `pedidos` e `itens_pedido`.

**Pre-requisitos:**
- Docker Compose rodando (`docker compose up -d`)
- SQL Server acessivel na porta `1433`
- Driver ODBC 18 instalado

## 1. Configuracao e conexao

In [ ]:
import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv

load_dotenv(override=True)

DB_SERVER = os.getenv('DB_SERVER')
DB_PORT = os.getenv('DB_PORT')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE', 'LojaDB')

print(f'Servidor: {DB_SERVER}:{DB_PORT}')
print(f'Database: {DB_DATABASE}')

In [ ]:
conn_master = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor_master = conn_master.cursor()
print('Conectado ao SQL Server (master).')

## 2. Criar database

In [ ]:
cursor_master.execute(f"""
    IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{DB_DATABASE}')
    BEGIN
        CREATE DATABASE [{DB_DATABASE}]
    END
""")

print(f'Database [{DB_DATABASE}] criado/verificado com sucesso.')
cursor_master.close()
conn_master.close()

In [ ]:
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor = conn.cursor()
print(f'Conectado ao [{DB_DATABASE}] com sucesso.')

## 3. Criar tabelas

In [ ]:
ddl_statements = [
    """
    IF OBJECT_ID('dbo.itens_pedido', 'U') IS NOT NULL DROP TABLE dbo.itens_pedido;
    IF OBJECT_ID('dbo.pedidos', 'U') IS NOT NULL DROP TABLE dbo.pedidos;
    IF OBJECT_ID('dbo.produtos', 'U') IS NOT NULL DROP TABLE dbo.produtos;
    IF OBJECT_ID('dbo.clientes', 'U') IS NOT NULL DROP TABLE dbo.clientes;
    """,
    """
    CREATE TABLE dbo.clientes (
        id INT PRIMARY KEY,
        nome VARCHAR(200) NOT NULL,
        email VARCHAR(200),
        telefone VARCHAR(30),
        cidade VARCHAR(100),
        estado VARCHAR(100),
        data_cadastro DATE
    )
    """,
    """
    CREATE TABLE dbo.produtos (
        id INT PRIMARY KEY,
        nome_produto VARCHAR(200) NOT NULL,
        categoria VARCHAR(100),
        preco DECIMAL(10,2),
        estoque INT,
        ativo BIT
    )
    """,
    """
    CREATE TABLE dbo.pedidos (
        id INT PRIMARY KEY,
        cliente_id INT NOT NULL,
        data_pedido DATE,
        status VARCHAR(30),
        valor_total DECIMAL(10,2),
        CONSTRAINT fk_pedidos_clientes FOREIGN KEY (cliente_id) REFERENCES dbo.clientes(id)
    )
    """,
    """
    CREATE TABLE dbo.itens_pedido (
        id INT PRIMARY KEY,
        pedido_id INT NOT NULL,
        produto_id INT NOT NULL,
        quantidade INT,
        preco_unitario DECIMAL(10,2),
        CONSTRAINT fk_itens_pedido_pedidos FOREIGN KEY (pedido_id) REFERENCES dbo.pedidos(id),
        CONSTRAINT fk_itens_pedido_produtos FOREIGN KEY (produto_id) REFERENCES dbo.produtos(id)
    )
    """
]

for ddl in ddl_statements:
    cursor.execute(ddl)

print('Tabelas criadas com sucesso.')

## 4. Carregar dados dos CSVs

In [ ]:
tabelas = ['clientes', 'produtos', 'pedidos', 'itens_pedido']
data_dir = os.path.join(os.getcwd(), 'data')

date_columns = {
    'clientes': ['data_cadastro'],
    'pedidos': ['data_pedido']
}

bool_columns = {
    'produtos': ['ativo']
}

for tabela in tabelas:
    csv_path = os.path.join(data_dir, f'{tabela}.csv')
    df = pd.read_csv(csv_path)

    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('string').str.strip()

    for col in date_columns.get(tabela, []):
        df[col] = pd.to_datetime(df[col], errors='coerce').dt.date

    for col in bool_columns.get(tabela, []):
        df[col] = df[col].map({'true': 1, 'false': 0, True: 1, False: 0})

    df = df.where(pd.notna(df), None)

    cols = ', '.join(df.columns)
    placeholders = ', '.join(['?' for _ in df.columns])
    insert_sql = f'INSERT INTO dbo.{tabela} ({cols}) VALUES ({placeholders})'
    data = [tuple(row) for row in df.itertuples(index=False, name=None)]

    cursor.fast_executemany = True
    cursor.executemany(insert_sql, data)
    print(f'{tabela}: {len(data)} registros inseridos')

print('Carga de dados concluida.')

## 5. Validacao

In [ ]:
print(f'{"Tabela":<20} {"Registros":>10}')
print('-' * 32)

for tabela in tabelas:
    cursor.execute(f'SELECT COUNT(*) FROM dbo.{tabela}')
    count = cursor.fetchone()[0]
    print(f'{tabela:<20} {count:>10}')

print('Validacao concluida.')

In [ ]:
for tabela in tabelas:
    print(f'
--- {tabela.upper()} (primeiros 5 registros) ---')
    df_sample = pd.read_sql(f'SELECT TOP 5 * FROM dbo.{tabela}', conn)
    print(df_sample.to_string(index=False))

In [ ]:
cursor.close()
conn.close()
print('Conexao encerrada.')